# Localized KG Drug Repurposing - Data Collection

This notebook is designed to run on **Kaggle** to fetch Secondary Data for building a global Knowledge Graph for Non-Small Cell Lung Carcinoma (NSCLC).
It fetches data from:
1. **Open Targets**: Gene-Disease Associations
2. **STRING API**: Protein-Protein Interactions
3. **ChEMBL API**: Drug-Target Interactions

In [ ]:
import requests
import pandas as pd
import json
import os
import time

# Create data directories
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

## 1. Fetch Open Targets Data (NSCLC Targets)

In [ ]:
def fetch_nsclc_targets():
    print("Fetching Target-Disease associations for NSCLC from Open Targets...")
    query = """
    query lungCancerTargets {
      disease(efoId: "MONDO_0005233") {
        id
        name
        associatedTargets(page: {index: 0, size: 1000}) {
          count
          rows {
            target {
              id
              approvedSymbol
              approvedName
              proteinIds {
                id
                source
              }
            }
            score
          }
        }
      }
    }
    """
    url = "https://api.platform.opentargets.org/api/v4/graphql"
    response = requests.post(url, json={"query": query})
    
    if response.status_code == 200:
        data = response.json()
        targets = data['data']['disease']['associatedTargets']['rows']
        
        target_list = []
        for t in targets:
            target_info = t['target']
            uniprot_id = None
            if target_info.get('proteinIds'):
                for pid in target_info['proteinIds']:
                    if pid['source'] == 'uniprot':
                        uniprot_id = pid['id']
                        break
                        
            if uniprot_id: # Only keep targets with Uniprot IDs for STRING matching
                target_list.append({
                    "disease_id": "MONDO_0005233",
                    "target_ensembl_id": target_info['id'],
                    "target_symbol": target_info['approvedSymbol'],
                    "uniprot_id": uniprot_id,
                    "association_score": t['score']
                })
            
        df = pd.DataFrame(target_list)
        output_path = "data/raw/opentargets_nsclc_targets.csv"
        df.to_csv(output_path, index=False)
        print(f"Saved {len(df)} targets to {output_path}")
        return df
    else:
        print(f"Failed: {response.status_code}")
        return None

targets_df = fetch_nsclc_targets()
targets_df.head()

## 2. Fetch Protein-Protein Interactions (STRING API)

In [ ]:
def fetch_string_ppi(uniprot_ids):
    print("Fetching Protein-Protein Interactions from STRING...")
    # STRING API limits to ~2000 proteins per request, we will chunk if needed
    string_api_url = "https://version-12-0.string-db.org/api/json/network"
    
    # We'll just take top 400 targets by score to avoid huge API requests in this demo
    ids_str = "%0d".join(uniprot_ids[:400]) 
    
    params = {
        "identifiers": ids_str,
        "species": 9606, # Human
        "caller_identity": "kaggle_kg_project"
    }
    
    response = requests.post(string_api_url, data=params)
    if response.status_code == 200:
        ppi_data = response.json()
        df = pd.DataFrame(ppi_data)
        df = df[['preferredName_A', 'preferredName_B', 'score']]
        output_path = "data/raw/string_ppi.csv"
        df.to_csv(output_path, index=False)
        print(f"Saved {len(df)} interactions to {output_path}")
        return df
    else:
        print(f"Failed STRING request: {response.status_code}")
        return None

ppi_df = fetch_string_ppi(targets_df['uniprot_id'].tolist())
ppi_df.head()

## 3. Fetch Drug-Target Interactions (ChEMBL API)

In [ ]:
def fetch_chembl_drugs(uniprot_ids):
    print("Fetching Drug-Target Interactions from ChEMBL...")
    drug_target_list = []
    
    # Fetch drugs for the top 50 targets (to save time in Kaggle execution)
    # In production, run for all targets
    for uid in uniprot_ids[:50]:
        url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism?target_component__accession={uid}&format=json"
        try:
            res = requests.get(url)
            if res.status_code == 200:
                mechs = res.json().get('mechanisms', [])
                for m in mechs:
                    drug_target_list.append({
                        "uniprot_id": uid,
                        "chembl_molecule_id": m['molecule_chembl_id'],
                        "action_type": m['action_type']
                    })
        except Exception as e:
            pass
        time.sleep(0.2) # Rate limiting
        
    df = pd.DataFrame(drug_target_list)
    output_path = "data/raw/chembl_drugs.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved {len(df)} drug-target pairs to {output_path}")
    return df

drug_df = fetch_chembl_drugs(targets_df['uniprot_id'].tolist())
drug_df.head()